# Laboratorio 7  NLP end-to-end y embeddings

+ Ana Laura Tschen 221645

In [20]:
from pathlib import Path
import sys, time, json, gc, platform
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset
import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, KeyedVectors
from gensim.test.utils import datapath

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import *
from src.preprocessing import *
from src.vocabulary import *
from src.sampling import *
from src.sgns import *
from src.evaluation import *
from src.analogies import *
from src.classification import *
from src.utils import *
seed_everything(SEED); ensure_dirs()
print(environment_report())
print('Dispositivo de entrenamiento:', DEVICE)
print('Pares: generación lazy en CPU; batches: transferencia al dispositivo seleccionado')

{'python': '3.13.2 | packaged by Anaconda, Inc. | (main, Feb  6 2025, 18:49:14) [MSC v.1929 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26200-SP0', 'torch': '2.14.0+cu126', 'cuda_available': True, 'gpu': 'NVIDIA GeForce RTX 3050', 'ram_gb': 95.72}
Dispositivo de entrenamiento: cuda
Pares: generación lazy en CPU; batches: transferencia al dispositivo seleccionado


## 2. Carga de datos

In [21]:
corpus = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1')
news = load_dataset('fancyzhx/ag_news')
glove = api.load('glove-wiki-gigaword-100')
print(corpus, news, len(glove.key_to_index))
print(corpus['train'][0], news['train'][0])

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1801350
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
}) DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
}) 400000
{'text': ''} {'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


## 3. Exploración y preprocesamiento 

In [22]:
raw_lines = corpus['train']['text']
tokens = list(tokenize_lines(raw_lines))
stats = corpus_stats(raw_lines, tokens)
display(pd.DataFrame([stats]))
print('Criterio de artículo: líneas de título que comienzan con espacio y terminan en =.')

,articles,lines,tokens,unique_tokens
0,0,1801350,105469355,483823


Criterio de artículo: líneas de título que comienzan con espacio y terminan en =.


In [23]:
examples = ["I can't re-use 3.14-year-old data.", "Dr. Smith's model—well, it works!", "They've paid $10.50.", "state-of-the-art NLP", "U.S.A. vs. UK", "rock 'n' roll", "Version 2.0-beta", "Cats, dogs... and birds?", "The 1990s weren't easy.", "e-mail addresses@example.com"]
tokenizer_table = tokenizer_comparison(examples)
tokenizer_table.to_csv(ARTIFACTS/'tables'/'tokenizer_comparison.csv', index=False)
display(tokenizer_table)

,oracion,tokens_custom,tokens_library,diferencias
0,I can't re-use 3.14-year-old data.,"[i, can't, re, -, use, 3.14, -, year, -, old, ...","[i, can, ', t, re, -, use, 3, ., 14, -, year, ...",True
1,"Dr. Smith's model—well, it works!","[dr, ., smith's, model, —, well, ,, it, works, !]","[dr, ., smith, ', s, model, —, well, ,, it, wo...",True
2,They've paid $10.50.,"[they've, paid, $, 10.50, .]","[they, ', ve, paid, $, 10, ., 50, .]",True
3,state-of-the-art NLP,"[state, -, of, -, the, -, art, nlp]","[state, -, of, -, the, -, art, nlp]",False
4,U.S.A. vs. UK,"[u, ., s, ., a, ., vs, ., uk]","[u, ., s, ., a, ., vs, ., uk]",False
5,rock 'n' roll,"[rock, ', n, ', roll]","[rock, ', n, ', roll]",False
6,Version 2.0-beta,"[version, 2.0, -, beta]","[version, 2, ., 0, -, beta]",True
7,"Cats, dogs... and birds?","[cats, ,, dogs, ., ., ., and, birds, ?]","[cats, ,, dogs, ..., and, birds, ?]",True
8,The 1990s weren't easy.,"[the, 1990, s, weren't, easy, .]","[the, 1990s, weren, ', t, easy, .]",True
9,e-mail addresses@example.com,"[e, -, mail, addresses, @, example, ., com]","[e, -, mail, addresses, @, example, ., com]",False


In [24]:
coverage = frequency_table(tokens)
coverage.to_csv(ARTIFACTS/'tables'/'top_k_coverage.csv', index=False)
display(coverage)
zipf_plot(tokens, ARTIFACTS/'figures'/'zipf.png')
display(pd.Series(Counter(tokens).most_common(20), name='count'))

,top_k,tokens_cubiertos,porcentaje
0,10,30799514,29.202335
1,1000,74528529,70.663681
2,30000,101708731,96.434392


0     (the, 6439062)
1       (,, 5120596)
2       (., 4172969)
3      (of, 2743121)
4     (and, 2505797)
5       (@, 2377841)
6      (in, 2177063)
7      (to, 1995098)
8       (a, 1801368)
9       (=, 1466599)
10      (", 1338685)
11    (was, 1076126)
12       (', 969215)
13       (-, 899487)
14       (s, 851744)
15      (on, 769693)
16      (as, 726091)
17     (for, 699803)
18    (that, 691217)
19    (with, 650774)
Name: count, dtype: object

In [25]:
mc = min_count_table(tokens, (2, 5, 10))
mc.to_csv(ARTIFACTS/'tables'/'min_count.csv', index=False)
display(mc)
MIN_COUNT = 5
vocab = Vocabulary.from_tokens(tokens, min_count=MIN_COUNT, max_size=30_000)
print('vocabulario elegido:', len(vocab.itos))

,min_count,vocab_size,known_tokens,oov_tokens,oov_percent
0,2,289013,105274545,194810,0.184708
1,5,165111,104947450,521905,0.494840
2,10,113861,104611839,857516,0.813048


vocabulario elegido: 30000


In [26]:
subsampling_table, sampled_tokens = subsampling_report(tokens, 1e-5, seed=SEED)
subsampling_table.to_csv(ARTIFACTS/'tables'/'subsampling.csv', index=False)
display(subsampling_table)
print('Pares teóricos sin subsampling:', pair_count(tokens, 5))
print('Pares teóricos con subsampling:', pair_count(sampled_tokens, 5))
pd.DataFrame([{'subsampling': False, 'pairs': pair_count(tokens,5)}, {'subsampling': True, 'pairs': pair_count(sampled_tokens,5)}]).to_csv(ARTIFACTS/'tables'/'skipgram_pairs.csv', index=False)

,word,frequency_original,discard_probability,appearances_before,appearances_kept,percent_discarded
0,the,6439062,0.987202,6439062,82325,98.721475
1,of,2743121,0.980392,2743121,53983,98.032059
2,and,2505797,0.979484,2505797,51295,97.952947


Pares teóricos sin subsampling: 1054693490
Pares teóricos con subsampling: 285277070


## 5. Experimentos de embeddings

In [27]:
RUN_EXPENSIVE = True
experiments = pd.DataFrame([
 {'experiment_id':'E20M_d50','corpus_tokens':20_000_000,'embedding_dim':50},
 {'experiment_id':'E20M_d100','corpus_tokens':20_000_000,'embedding_dim':100},
 {'experiment_id':'E20M_d300','corpus_tokens':20_000_000,'embedding_dim':300},
 {'experiment_id':'E50pct_d100','corpus_tokens':10_000_000,'embedding_dim':100},
 {'experiment_id':'E100pct_d100','corpus_tokens':20_000_000,'embedding_dim':100},
])
experiments.to_csv(ARTIFACTS/'tables'/'experiment_configurations.csv', index=False)
display(experiments)

,experiment_id,corpus_tokens,embedding_dim
0,E20M_d50,20000000,50
1,E20M_d100,20000000,100
2,E20M_d300,20000000,300
3,E50pct_d100,10000000,100
4,E100pct_d100,20000000,100


In [ ]:
# SGNS propio: dos tablas nn.Embedding + negative sampling p(w) proporcional a count(w)^0.75
# Se recorta por configuración, nunca por debajo de 20M para experimentos declarados como principales.
if RUN_EXPENSIVE:
    main_tokens = tokens[:20_000_000]
    ids = vocab.encode(main_tokens)
    distribution = negative_distribution(vocab.counts)
    for row in experiments.itertuples():
        cfg = ExperimentConfig(experiment_id=row.experiment_id, corpus_tokens=row.corpus_tokens, embedding_dim=row.embedding_dim)
        selected = ids[:cfg.corpus_tokens]
        pairs = generate_skipgram_pairs(selected, cfg.window_size, SEED)
        model = SGNS(len(vocab.itos), cfg.embedding_dim)
        history = train_sgns(model, pairs, distribution, cfg, DEVICE, ARTIFACTS/'checkpoints'/f'{cfg.experiment_id}.pt', RESULTS/f'{cfg.experiment_id}_history.csv')
else:
    print("turn expensive to true")

E20M_d50 epoch 1: 0it [00:00, ?it/s]

In [ ]:
# Word2Vec gensim con condiciones equivalentes al experimento seleccionado
# El test oficial de AG News y SimLex no participan en la selección.
if RUN_EXPENSIVE:
    sentence_tokens = [tokenize_custom(line) for line in raw_lines if line.strip()]
    w2v = Word2Vec(sentence_tokens, vector_size=100, window=5, min_count=MIN_COUNT, sg=1, negative=5, sample=1e-5, epochs=3, seed=SEED, workers=1)
    w2v.wv.save(ARTIFACTS/'embeddings'/'word2vec_gensim.kv')
else:
    print(' RUN_EXPENSIVE.')

Word2Vec omitido hasta activar RUN_EXPENSIVE.


## 6. Evaluación intrínseca, aritmética y paralelismo.

In [ ]:
control_words = ['king','france','computer','good','january','run']
analogies_own = [('man','woman','king','gender'),('paris','france','rome','country-capital'),('italy','italian','spain','country-demonym'),('good','better','bad','comparative'),('good','best','bad','superlative'),('walk','walking','run','tense'),('cat','cats','dog','plural'),('january','february','march','time'),('boy','girl','king','gender'),('london','england','berlin','country-capital'),('slow','slower','fast','comparative'),('small','smallest','large','superlative'),('swim','swimming','run','tense'),('book','books','car','plural'),('monday','tuesday','friday','time')]
print('Analogías preparadas:', len(analogies_own), 'en siete tipos.')

Analogías preparadas: 15 en siete tipos.


## 7. t-SNE

In [ ]:
# La celda se activa tras disponer del mejor SGNS y sklearn.manifold.TSNE.
print('Figura t-SNE: ejecutar después de entrenar y seleccionar best_model.pt.')

Figura t-SNE: ejecutar después de entrenar y seleccionar best_model.pt.


## 8. AG News y clasificadores


In [ ]:
from sklearn.model_selection import train_test_split
news_text = [normalize_text(x) for x in news['train']['text']]
news_y = np.asarray(news['train']['label'])
train_x, val_x, train_y, val_y = train_test_split(news_text, news_y, test_size=.10, random_state=SEED, stratify=news_y)
news_test_x = [normalize_text(x) for x in news['test']['text']]
news_test_y = np.asarray(news['test']['label'])
print(len(train_x), len(val_x), len(news_test_x))
tfidf_result, tfidf_model, tfidf_vectorizer = tfidf_baseline(train_x, train_y, val_x, val_y)
display(pd.DataFrame([tfidf_result]))

108000 12000 7600


,accuracy,macro_precision,macro_recall,macro_f1,model,training_time,parameters
0,0.923083,0.922856,0.923083,0.922857,tfidf_logistic_regression,26.794252,400004


In [ ]:
fractions = [0.01, 0.10, 0.25, 0.50, 1.00]
print('Experimento de fracciones preparado:', fractions)
print('Matrices de confusión y classification_report se guardan para cada modelo final ejecutado.')

Experimento de fracciones preparado: [0.01, 0.1, 0.25, 0.5, 1.0]
Matrices de confusión y classification_report se guardan para cada modelo final ejecutado.


## 9. Tabla comparativa 

In [ ]:
# Auditoría final de artefactos
print('Figuras:', sorted(str(p) for p in (ARTIFACTS/'figures').glob('*')))
print('Tablas:', sorted(str(p) for p in (ARTIFACTS/'tables').glob('*')))
print('Resultados:', sorted(str(p) for p in RESULTS.glob('*')))

Figuras: ['C:\\Users\\alambre1\\Documents\\UVG\\2026\\2026s2\\deep_learning\\lab7\\artifacts\\figures\\zipf.png']
Tablas: ['C:\\Users\\alambre1\\Documents\\UVG\\2026\\2026s2\\deep_learning\\lab7\\artifacts\\tables\\experiment_configurations.csv', 'C:\\Users\\alambre1\\Documents\\UVG\\2026\\2026s2\\deep_learning\\lab7\\artifacts\\tables\\min_count.csv', 'C:\\Users\\alambre1\\Documents\\UVG\\2026\\2026s2\\deep_learning\\lab7\\artifacts\\tables\\skipgram_pairs.csv', 'C:\\Users\\alambre1\\Documents\\UVG\\2026\\2026s2\\deep_learning\\lab7\\artifacts\\tables\\subsampling.csv', 'C:\\Users\\alambre1\\Documents\\UVG\\2026\\2026s2\\deep_learning\\lab7\\artifacts\\tables\\tokenizer_comparison.csv', 'C:\\Users\\alambre1\\Documents\\UVG\\2026\\2026s2\\deep_learning\\lab7\\artifacts\\tables\\top_k_coverage.csv']
Resultados: []
